# Distribution Sandbox

*Companion notebook to* **Chapter 1, §1.2** *of* Mathemagics: A Field Guide to Random Matrices and Spectral Clustering.

Generate samples from the seven standard distributions and watch their empirical histograms converge to the theoretical PMFs/PDFs as the sample size $N$ grows.

**How to use:** run each cell top to bottom. Change parameters to taste; the plots refresh automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=2026)
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

# Mathemagics palette
C = {
    'def': '#5A9CCB', 'intu': '#B8901C', 'recap': '#50A076',
    'aside': '#C57A5C', 'where': '#8273A8', 'colab': '#D4A028',
    'thm': '#2D2D2D',
}

## Discrete distributions

Each row: empirical histogram (left) versus theoretical PMF (right). Try different $N$ to see how the two converge.

In [ ]:
def plot_discrete(samples, theoretical_pmf, support, title, color):
    fig, ax = plt.subplots(figsize=(6, 2.4))
    counts = np.bincount(samples, minlength=support[-1] + 1)
    emp = counts[support[0]:support[-1] + 1] / len(samples)
    th = theoretical_pmf(support)
    w = 0.35
    ax.bar(support - w/2, emp, width=w, color=color, alpha=0.85,
           edgecolor=C['thm'], lw=0.4, label='empirical')
    ax.bar(support + w/2, th, width=w, color=C['intu'], alpha=0.85,
           edgecolor=C['thm'], lw=0.4, label='theoretical')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('value $k$'); ax.set_ylabel('probability')
    ax.legend(frameon=False, fontsize=8)
    plt.show()

In [ ]:
# Bernoulli(p) -- two outcomes
p = 0.3
N = 5_000
samples = rng.binomial(1, p, size=N)
support = np.array([0, 1])
plot_discrete(samples, lambda k: stats.bernoulli.pmf(k, p), support,
              f'Bernoulli($p={p}$),  $N={N:,}$', C['def'])

In [ ]:
# Binomial(n, p) -- sum of n independent Bernoulli(p) trials
n, p = 20, 0.3
N = 5_000
samples = rng.binomial(n, p, size=N)
support = np.arange(0, n + 1)
plot_discrete(samples, lambda k: stats.binom.pmf(k, n, p), support,
              f'Binomial($n={n}, p={p}$),  $N={N:,}$', C['def'])

In [ ]:
# Poisson(lambda) -- count of rare events
lam = 4.0
N = 5_000
samples = rng.poisson(lam, size=N)
support = np.arange(0, samples.max() + 1)
plot_discrete(samples, lambda k: stats.poisson.pmf(k, lam), support,
              f'Poisson($\\lambda={lam}$),  $N={N:,}$', C['def'])

## Continuous distributions

Empirical histogram (filled bars) versus theoretical PDF (smooth curve).

In [ ]:
def plot_continuous(samples, pdf, x_range, title, color):
    fig, ax = plt.subplots(figsize=(6, 2.4))
    ax.hist(samples, bins=40, density=True, color=color, alpha=0.55,
            edgecolor=C['thm'], lw=0.3, label='empirical')
    xs = np.linspace(x_range[0], x_range[1], 400)
    ax.plot(xs, pdf(xs), color=C['aside'], lw=1.5, label='theoretical')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('$x$'); ax.set_ylabel('density')
    ax.legend(frameon=False, fontsize=8)
    plt.show()

In [ ]:
# Uniform on [0, 1]
N = 5_000
samples = rng.uniform(0, 1, size=N)
plot_continuous(samples, lambda x: stats.uniform.pdf(x), (-0.2, 1.2),
                f'Uniform on $[0, 1]$,  $N={N:,}$', C['recap'])

In [ ]:
# Normal(mu, sigma^2) -- the bell curve
mu, sigma = 0, 1
N = 5_000
samples = rng.normal(mu, sigma, size=N)
plot_continuous(samples,
                lambda x: stats.norm.pdf(x, mu, sigma),
                (-4, 4),
                f'Normal $\\mathcal{{N}}({mu}, {sigma}^2)$,  $N={N:,}$',
                C['recap'])

In [ ]:
# Exponential(lambda)
lam = 1.0
N = 5_000
samples = rng.exponential(scale=1.0/lam, size=N)
plot_continuous(samples,
                lambda x: stats.expon.pdf(x, scale=1.0/lam),
                (0, 6),
                f'Exponential($\\lambda={lam}$),  $N={N:,}$',
                C['recap'])

## Convergence demo

Watch the empirical histogram of a $\mathrm{Bin}(20, 0.3)$ sample close in on the theoretical PMF as $N$ grows from $50$ to $50\,000$.

In [ ]:
n, p = 20, 0.3
fig, axes = plt.subplots(1, 4, figsize=(11, 2.4), sharey=True)
support = np.arange(0, n + 1)
th = stats.binom.pmf(support, n, p)
for ax, N in zip(axes, [50, 500, 5_000, 50_000]):
    samples = rng.binomial(n, p, size=N)
    emp = np.bincount(samples, minlength=n + 1) / N
    ax.bar(support, emp, color=C['def'], alpha=0.6, edgecolor=C['thm'], lw=0.3)
    ax.plot(support, th, 'o-', color=C['aside'], markersize=3, lw=0.8,
            label='theoretical')
    ax.set_title(f'$N = {N:,}$', fontsize=10)
    ax.set_xlabel('$k$')
axes[0].set_ylabel('probability')
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Try it yourself

- Change `N` and watch the empirical bars get closer to the theoretical PMF/PDF.
- Pick a Poisson with $\lambda = 0.3$ vs $\lambda = 30$ and see how the shape changes.
- For the Normal, what does $\mathcal{N}(5, 0.5^2)$ look like compared to $\mathcal{N}(0, 1)$? (Hint: shifted mean, narrower spread.)